In [1]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

def scrape_ambitionbox():
    url = "https://www.ambitionbox.com/list-of-companies?campaign=desktop_navpage%3D1"
    
    # Configure Selenium Chrome options to look like a human browser
    chrome_options = Options()
    # chrome_options.add_argument("--headless")  # Uncomment to run in background once confirmed working
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("window-size=1280,800")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    # Initialize the browser driver
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)
    
    try:
        print("Opening AmbitionBox...")
        driver.get(url)
        
        # Give the page 8 seconds to fully execute scripts and bypass any landing filters
        time.sleep(8)
        
        # Pull down the rendered HTML source
        page_source = driver.page_source
        soup = BeautifulSoup(page_source, "html.parser")
        
        companies_list = []
        
        # Target the h3 tags directly, which contain the core company blocks
        h3_tags = soup.find_all("h3")
        print(f"Found {len(h3_tags)} potential listings. Processing data...")
        
        for h3 in h3_tags:
            name = h3.text.strip()
            
            # Clean up out-of-bounds page texts that use h3 elements
            if name in ["Top Companies in INDIA", "Companies in India", "Get the AmbitionBox App"]:
                continue
                
            # Ascend up to the main section containing rating and review data
            parent = h3.find_parent("div")
            rating = "N/A"
            reviews = "N/A"
            industry_info = "N/A"
            
            if parent:
                # Find the rating text next to the name
                # (AmbitionBox commonly groups ratings in strong elements, spans, or paragraphs)
                all_text = parent.get_text(separator="|", strip=True)
                text_parts = all_text.split('|')
                
                # Check for standard numerical ratings (e.g., '3.3', '3.7')
                for part in text_parts:
                    if part.replace('.', '', 1).isdigit() and len(part) <= 4:
                        rating = part
                        break
                
                # Find reviews (usually explicitly labeled or mapped in parentheses next to rating)
                review_links = parent.find_all("a")
                for link in review_links:
                    link_text = link.text.strip()
                    if "Reviews" in link_text or "k" in link_text:
                        reviews = link_text
                        break
            
            companies_list.append({
                "Company Name": name,
                "Rating": rating,
                "Reviews": reviews
            })
            
        # Compile into a clean DataFrame
        df = pd.DataFrame(companies_list)
        
        # Filter out accidental blank parses
        df = df[df["Company Name"] != ""]
        
        print("\n--- Scraped Company Details ---")
        print(df.to_string(index=False))
        
        
    except Exception as e:
        print(f"An error occurred: {e}")
        
    finally:
        # Securely shut down the automated browser instance
        driver.quit()

if __name__ == "__main__":
    scrape_ambitionbox()

ModuleNotFoundError: No module named 'selenium'